##### Boiler-Plate TensorFlow Model.

> this is a boiler plate code from a google tutorial:

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/site/en/tutorials/quickstart/beginner.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" /> run on google colab</a>
  </td>
</table>

built on [keras](https://www.tensorflow.org/guide/keras/overview):

1. load a prebuilt dataset.
1. `model`:neural network machine learning model that classifies images.
2. train this neural network.
3. evaluate the accuracy of it.

this is a [google colab](https://colab.research.google.com/notebooks/welcome.ipynb) notebook. use the browser as your frontend for your notebook.  
> connect to a python runtime:
 > run the code in the notebook, per cell(shift + enter) or all of them at once.


## set up `tensorflow`

import tensorflow into your program this way:

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

2025-10-20 16:33:27.417031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.20.0-dev0+selfbuilt


LOCAL SETUP: 
> sadface :}
> see the [install guide](https://www.tensorflow.org/install) for setting up `tensorflow` locally.

> note: make sure you have upgraded to the latest `pip` to install the tensorflow package
> or use your system's way for your setup eg on arch you can install it w/ the commandline below
>
>
```sh
sudo pacman -S python-tensorflow
```

## load MNIST dataset

load and prepare the MNIST dataset. the pixel values of the images range from 0 through 255. you can scale these values to a range of 0 to 1 by dividing the values by `255.0`. this also converts the sample data from integers to floating-point numbers:

In [2]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


## machine learning model

you can decide to go with a `tf.keras.Sequential` model:

In [3]:
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dropout(0.2),
  tf.keras.layers.Dense(10)
])

/usr/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


[`Sequential`](https://www.tensorflow.org/guide/keras/sequential_model) is useful for stacking layers where each layer has one input [tensor](https://www.tensorflow.org/guide/tensor) and one output tensor. layers are functions with a known mathematical structure that can be reused and have trainable variables. most models are composed of layers. this model uses the [`Flatten`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Flatten), [`Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense), and [`Dropout`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout) layers.

the model returns a vector of [logits](https://developers.google.com/machine-learning/glossary#logits) or [log-odds](https://developers.google.com/machine-learning/glossary#log-odds) scores, one for each class.

In [4]:
predictions = model(x_train[:1]).numpy()
predictions

array([[-0.11595236,  0.10088754, -0.3635874 ,  0.27606001,  0.01914483,
         0.25733286,  0.43249524, -0.05425551,  0.47111008,  0.29140648]],
      dtype=float32)

the `tf.nn.softmax` function converts these logits to *probabilities* for each class: 

In [5]:
tf.nn.softmax(predictions).numpy()

array([[0.07578263, 0.09413302, 0.05915931, 0.11215495, 0.08674444,
        0.11007416, 0.13114671, 0.08060542, 0.13630997, 0.1138894 ]],
      dtype=float32)

NOTE: It is possible to bake the `tf.nn.softmax` function into the activation function for the last layer of the network. while this can make the model output more directly interpretable, this approach is discouraged as it's impossible to provide an exact and numerically stable loss calculation for all models when using a softmax output. 

define a loss function for training using `losses.SparseCategoricalCrossentropy`:

In [6]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

the loss function takes a vector of ground truth values and a vector of logits and returns a scalar loss for each example. this loss is equal to the negative log probability of the true class: The loss is zero if the model is sure of the correct class.

this untrained model gives probabilities close to random (1/10 for each class), so the initial loss should be close to `-tf.math.log(1/10) ~= 2.3`.

In [7]:
loss_fn(y_train[:1], predictions).numpy()

np.float32(2.2066011)

before you start training, configure and compile the model using Keras `Model.compile`. Set the [`optimizer`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) class to `adam`, set the `loss` to the `loss_fn` function you defined earlier, and specify a metric to be evaluated for the model by setting the `metrics` parameter to `accuracy`.

In [8]:
model.compile(optimizer='adam',
              loss=loss_fn,
              metrics=['accuracy'])

## train + evaluate model

use the `Model.fit` method to adjust your model parameters and minimize the loss: 

In [9]:
model.fit(x_train, y_train, epochs=5)

Epoch 1/5


2025-10-20 16:35:05.353696: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 188160000 exceeds 10% of free system memory.


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9148 - loss: 0.2968
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9573 - loss: 0.1424
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9668 - loss: 0.1075
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9730 - loss: 0.0854
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9772 - loss: 0.0731


the `Model.evaluate` method checks the model's performance, usually on a [validation set](https://developers.google.com/machine-learning/glossary#validation-set) or [test set](https://developers.google.com/machine-learning/glossary#test-set).

In [10]:
model.evaluate(x_test,  y_test, verbose=2)

2025-10-20 16:35:44.290481: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 31360000 exceeds 10% of free system memory.


313/313 - 1s - 2ms/step - accuracy: 0.9775 - loss: 0.0712


[0.07123063504695892, 0.9775000214576721]

the image classifier is now trained to ~98% accuracy on this dataset

the model can return a probability, if can wrap the trained model, and attach the softmax to it:

In [11]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])

In [12]:
probability_model(x_test[:5])

<tf.Tensor: shape=(5, 10), dtype=float32, numpy=
array([[6.69583358e-07, 1.43414587e-08, 1.90333412e-05, 5.69479773e-04,
        1.11522465e-10, 2.12341575e-07, 1.68400450e-14, 9.99361932e-01,
        7.75794160e-06, 4.09996064e-05],
       [8.64385008e-09, 1.69053848e-04, 9.99820650e-01, 9.82498841e-06,
        4.96298891e-16, 4.96625091e-07, 4.17238954e-09, 9.83653263e-13,
        4.94363928e-09, 2.06067531e-15],
       [4.72314099e-07, 9.98238921e-01, 1.27257299e-04, 2.47044918e-05,
        5.86583628e-05, 4.60125693e-06, 1.27159574e-05, 1.39301794e-03,
        1.38289266e-04, 1.26694340e-06],
       [9.99970555e-01, 1.54768509e-09, 1.23427762e-05, 4.60671856e-07,
        1.13440046e-06, 1.65414212e-06, 6.41790791e-07, 1.57357931e-06,
        1.17918368e-08, 1.15816156e-05],
       [5.60763134e-08, 1.95119676e-09, 9.01271392e-07, 5.83164494e-09,
        9.99548018e-01, 2.79044599e-07, 1.67398738e-07, 2.04382686e-05,
        1.16354492e-07, 4.30057466e-04]], dtype=float32)>

## end